# CNN Spectrogram Classifier Training — ISTerre Event Classification

**Goal**: train a CNN to classify seismic event type (earthquake / rockslide / ice quake)
from 3-component (Z, N, E) spectrogram images built by `07a_spectrogram_dataset_build.py`
on the ISTerre cluster (that script does the SDS access + instrument response removal +
spectrogram computation — this notebook only trains, since the cluster has no GPU).

**Drive folder expected** (upload the run folder produced by 07a, renaming/reorganizing
as needed):
```
MyDrive/colab_cnn_training_spectrogram/
    images/            <- *.npz files, one per (event x station) sample
    image_list.csv     <- manifest: fname, event_time, event_type, network, station,
                          channel, det_starttime, split (train/val/test)
    freq_axis.npy       <- shared frequency axis [Hz]
    time_axis.npy       <- shared time axis [s]
```

**Runtime**: Runtime > Change runtime type > T4 GPU

## Cell 1 — Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print('No GPU — go to Runtime > Change runtime type > T4 GPU')

## Cell 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

## Cell 3 — Configure paths & hyperparameters
**Edit this cell** to match your Drive folder.

In [ ]:
import os

# -- Edit here -----------------------------------------------------------
DRIVE_BASE     = '/content/drive/MyDrive/colab_cnn_training_spectrogram'
IMAGES_DIR     = os.path.join(DRIVE_BASE, 'images')
MANIFEST_CSV   = os.path.join(DRIVE_BASE, 'image_list.csv')
FREQ_AXIS_PATH = os.path.join(DRIVE_BASE, 'freq_axis.npy')
TIME_AXIS_PATH = os.path.join(DRIVE_BASE, 'time_axis.npy')
LOG_DIR        = os.path.join(DRIVE_BASE, 'log')

# Must match TARGET_CLASSES in 07a_spectrogram_dataset_build.py
CLASS_NAMES = ['earthquake', 'rockslide', 'ice quake']

EPOCHS        = 60
BATCH_SIZE    = 32
LEARNING_RATE = 1e-3
DROPOUT_RATE  = 0.5
# --------------------------------------------------------------------------

os.makedirs(LOG_DIR, exist_ok=True)
for label, path in [('images/', IMAGES_DIR), ('image_list.csv', MANIFEST_CSV),
                    ('freq_axis.npy', FREQ_AXIS_PATH), ('time_axis.npy', TIME_AXIS_PATH)]:
    exists = os.path.exists(path)
    count  = f'  ({len(os.listdir(path))} files)' if exists and os.path.isdir(path) else ''
    print(f"{'OK' if exists else 'MISSING'}  {label:20s}  {path}{count}")

## Cell 4 — Check TensorFlow + GPU visibility

In [ ]:
import tensorflow as tf
print(f'TensorFlow : {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs       : {gpus}')
if not gpus:
    print('WARNING: no GPU — go to Runtime > Change runtime type > T4 GPU')

## Cell 5 — Load manifest and inspect class / split distribution

Splits (train/val/test) were already assigned by 07a on the cluster, by EVENT
(stratified by class) — this notebook just reads the `split` column, no re-splitting.

In [ ]:
import pandas as pd
import numpy as np

manifest = pd.read_csv(MANIFEST_CSV)
print(f'Manifest: {len(manifest):,} rows')

label2idx = {name: i for i, name in enumerate(CLASS_NAMES)}
idx2label = {i: name for name, i in label2idx.items()}
print('Label encoding:', label2idx)

print('\nRows per split x class:')
print(manifest.groupby(['split', 'event_type']).size().unstack(fill_value=0).to_string())

freq_axis = np.load(FREQ_AXIS_PATH)
time_axis = np.load(TIME_AXIS_PATH)
print(f'\nFrequency axis: {len(freq_axis)} bins, 0-{freq_axis.max():.1f} Hz')
print(f'Time axis     : {len(time_axis)} bins, 0-{time_axis.max():.1f} s')

## Cell 6 — Load spectrogram images into memory

Images are small (a few hundred KB each) — for a dataset in the hundreds-to-low-thousands
range, loading everything into RAM up front is simpler and faster than streaming from
Drive during training. If your dataset is much larger, switch this to a `tf.data.Dataset`
that reads .npz files lazily instead.

In [ ]:
def load_split(split_name):
    rows = manifest[manifest['split'] == split_name]
    images, labels = [], []
    for _, row in rows.iterrows():
        with np.load(os.path.join(IMAGES_DIR, row['fname'])) as d:
            images.append(d['image'].astype('float32'))
        labels.append(label2idx[row['event_type']])
    X = np.stack(images, axis=0) if images else np.empty((0,) + images[0].shape if images else (0,))
    y = np.array(labels, dtype='int32')
    return X, y

X_train, y_train = load_split('train')
X_val,   y_val   = load_split('val')
X_test,  y_test  = load_split('test')

print(f'Train: X={X_train.shape}  y={y_train.shape}')
print(f'Val  : X={X_val.shape}  y={y_val.shape}')
print(f'Test : X={X_test.shape}  y={y_test.shape}')

INPUT_SHAPE = X_train.shape[1:]
print(f'\nCNN input shape: {INPUT_SHAPE}')

## Cell 7 — Per-channel normalization (fit on TRAIN only)

Z-score each of the 3 channels (Z, N, E spectrograms) independently, using statistics
computed on the training set only — never fit normalization on val/test, that would leak
information. Stats are saved to Drive so the exact same normalization can be reused at
inference time on new spectrograms.

In [ ]:
channel_mean = X_train.mean(axis=(0, 1, 2), keepdims=True)
channel_std  = X_train.std(axis=(0, 1, 2), keepdims=True) + 1e-8

X_train_n = (X_train - channel_mean) / channel_std
X_val_n   = (X_val   - channel_mean) / channel_std
X_test_n  = (X_test  - channel_mean) / channel_std

np.savez(os.path.join(LOG_DIR, 'normalization_stats.npz'),
         mean=channel_mean, std=channel_std)
print('Per-channel mean:', channel_mean.ravel())
print('Per-channel std :', channel_std.ravel())
print('Saved -> normalization_stats.npz')

## Cell 8 — Data augmentation (SpecAugment-style) + tf.data pipeline

Replaces SMOTE (which doesn't map onto images): random time masking, random frequency
masking, and small amplitude jitter, applied only to the training set. This both
regularizes the small dataset and gives the CNN some invariance to exactly where in
time/frequency the informative structure sits.

In [ ]:
def spec_augment(image, label, n_time_masks=1, n_freq_masks=1,
                 max_time_mask_frac=0.15, max_freq_mask_frac=0.15, noise_std=0.05):
    img = tf.identity(image)
    n_freq  = tf.shape(img)[0]
    n_time  = tf.shape(img)[1]

    for _ in range(n_time_masks):
        mask_w = tf.random.uniform([], 0, tf.cast(tf.cast(n_time, tf.float32) * max_time_mask_frac, tf.int32) + 1, dtype=tf.int32)
        mask_w = tf.maximum(mask_w, 1)
        t0 = tf.random.uniform([], 0, tf.maximum(n_time - mask_w, 1), dtype=tf.int32)
        time_idx = tf.range(n_time)
        time_mask = tf.logical_and(time_idx >= t0, time_idx < t0 + mask_w)
        time_mask = tf.cast(tf.logical_not(time_mask), img.dtype)[tf.newaxis, :, tf.newaxis]
        img = img * time_mask

    for _ in range(n_freq_masks):
        mask_h = tf.random.uniform([], 0, tf.cast(tf.cast(n_freq, tf.float32) * max_freq_mask_frac, tf.int32) + 1, dtype=tf.int32)
        mask_h = tf.maximum(mask_h, 1)
        f0 = tf.random.uniform([], 0, tf.maximum(n_freq - mask_h, 1), dtype=tf.int32)
        freq_idx = tf.range(n_freq)
        freq_mask = tf.logical_and(freq_idx >= f0, freq_idx < f0 + mask_h)
        freq_mask = tf.cast(tf.logical_not(freq_mask), img.dtype)[:, tf.newaxis, tf.newaxis]
        img = img * freq_mask

    img = img + tf.random.normal(tf.shape(img), mean=0.0, stddev=noise_std, dtype=img.dtype)
    return img, label

AUTOTUNE = tf.data.AUTOTUNE

train_ds = (tf.data.Dataset.from_tensor_slices((X_train_n, y_train))
            .shuffle(buffer_size=len(X_train_n), seed=42)
            .map(spec_augment, num_parallel_calls=AUTOTUNE)
            .batch(BATCH_SIZE)
            .prefetch(AUTOTUNE))

val_ds = (tf.data.Dataset.from_tensor_slices((X_val_n, y_val))
          .batch(BATCH_SIZE)
          .prefetch(AUTOTUNE))

test_ds = (tf.data.Dataset.from_tensor_slices((X_test_n, y_test))
           .batch(BATCH_SIZE)
           .prefetch(AUTOTUNE))

print('tf.data pipelines ready.')

## Cell 9 — Build the CNN

Shallow on purpose: this dataset is small (hundreds-to-low-thousands of samples), so a
deep network would overfit fast. GlobalAveragePooling2D instead of Flatten+Dense keeps
the parameter count low. Strided convolutions handle downsampling (no separate pooling
layers), same style as the depthwise conv blocks in `deepdenoiser/model.py`.

In [ ]:
from tensorflow.keras import layers, models

def build_cnn(input_shape, n_classes, dropout_rate=0.5):
    inputs = layers.Input(shape=input_shape, name='spectrogram')

    x = inputs
    for i, filters in enumerate([32, 64, 128]):
        x = layers.Conv2D(filters, 3, padding='same', use_bias=False, name=f'conv{i+1}_a')(x)
        x = layers.BatchNormalization(name=f'bn{i+1}_a')(x)
        x = layers.Activation('relu', name=f'relu{i+1}_a')(x)
        x = layers.Conv2D(filters, 3, strides=2, padding='same', use_bias=False, name=f'conv{i+1}_down')(x)
        x = layers.BatchNormalization(name=f'bn{i+1}_down')(x)
        x = layers.Activation('relu', name=f'relu{i+1}_down')(x)

    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dropout(dropout_rate, name='dropout')(x)
    outputs = layers.Dense(n_classes, activation='softmax', name='predictions')(x)

    return models.Model(inputs, outputs, name='spectrogram_cnn')

model = build_cnn(INPUT_SHAPE, len(CLASS_NAMES), dropout_rate=DROPOUT_RATE)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

## Cell 10 — Class weights + callbacks

`class_weight='balanced'` is the CNN-training analog of `RF_CLASS_WEIGHT='balanced'` in
`06a_train_RF_classifier.py` — SMOTE doesn't apply to images, so this (plus the
augmentation in Cell 8) is how class imbalance gets handled here.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weight_values = compute_class_weight('balanced', classes=np.arange(len(CLASS_NAMES)), y=y_train)
class_weight_dict = {i: w for i, w in enumerate(class_weight_values)}
print('Class weights:', {idx2label[i]: round(w, 3) for i, w in class_weight_dict.items()})

checkpoint_path = os.path.join(LOG_DIR, 'best_model.keras')
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(checkpoint_path, monitor='val_loss', save_best_only=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6),
]
print(f'Checkpoints -> {checkpoint_path}')

## Cell 11 — Train

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weight_dict,
    callbacks=callbacks,
)

## Cell 12 — Training curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(history.history['loss'], label='train')
axes[0].plot(history.history['val_loss'], label='val')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].set_title('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['accuracy'], label='train')
axes[1].plot(history.history['val_accuracy'], label='val')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy'); axes[1].set_title('Accuracy')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
curves_path = os.path.join(LOG_DIR, 'training_curves.png')
plt.savefig(curves_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'[SAVED] {curves_path}')

## Cell 13 — Evaluate on test set

Same evaluation suite as `06a_train_RF_classifier.py`: classification report, confusion
matrix, one-vs-rest ROC curves — so CNN and RF/HGB results are directly comparable.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
from sklearn.preprocessing import label_binarize

y_proba = model.predict(test_ds)
y_pred  = np.argmax(y_proba, axis=1)

print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_test, y_pred, labels=np.arange(len(CLASS_NAMES)))
print('Confusion matrix (rows=true, cols=predicted):')
print(pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_string())

fig_cm, ax_cm = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES).plot(ax=ax_cm, cmap='Blues', colorbar=False)
ax_cm.set_title('Confusion matrix — test set (CNN)')
plt.tight_layout()
cm_path = os.path.join(LOG_DIR, 'confusion_matrix.png')
plt.savefig(cm_path, dpi=150)
plt.show()
print(f'[SAVED] {cm_path}')

y_test_bin = label_binarize(y_test, classes=np.arange(len(CLASS_NAMES)))
fig_roc, ax_roc = plt.subplots(figsize=(7, 5))
for i, name in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
    ax_roc.plot(fpr, tpr, lw=2, label=f'{name}  (AUC={auc(fpr, tpr):.3f})')
ax_roc.plot([0, 1], [0, 1], 'k--', lw=1)
ax_roc.set_xlabel('False Positive Rate'); ax_roc.set_ylabel('True Positive Rate')
ax_roc.set_title('ROC curves — one-vs-rest (test set, CNN)')
ax_roc.legend(loc='lower right')
plt.tight_layout()
roc_path = os.path.join(LOG_DIR, 'roc_curves.png')
plt.savefig(roc_path, dpi=150)
plt.show()
print(f'[SAVED] {roc_path}')

## Cell 14 — Grad-CAM on example test images

Shows which time-frequency region the CNN is keying on for a few correctly-classified
examples per class — a domain-expert sanity check that the network is reading actual
event signatures (e.g. a rockslide's low-frequency coda) rather than an artifact.

In [ ]:
last_conv_name = None
for layer in model.layers:
    if isinstance(layer, layers.Conv2D):
        last_conv_name = layer.name
print(f'Grad-CAM target layer: {last_conv_name}')

grad_model = tf.keras.models.Model(model.inputs, [model.get_layer(last_conv_name).output, model.output])

def grad_cam(img_batch, class_idx):
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_batch)
        loss = preds[:, class_idx]
    grads = tape.gradient(loss, conv_out)
    weights = tf.reduce_mean(grads, axis=(1, 2), keepdims=True)
    cam = tf.reduce_sum(weights * conv_out, axis=-1)
    cam = tf.nn.relu(cam)
    cam = cam / (tf.reduce_max(cam, axis=(1, 2), keepdims=True) + 1e-8)
    return cam.numpy()

fig, axes = plt.subplots(len(CLASS_NAMES), 2, figsize=(9, 3.2 * len(CLASS_NAMES)))
for row, name in enumerate(CLASS_NAMES):
    idx_candidates = np.where((y_test == label2idx[name]) & (y_pred == label2idx[name]))[0]
    if len(idx_candidates) == 0:
        for col in range(2):
            axes[row, col].text(0.5, 0.5, f'No correct\n{name} example', ha='center', va='center')
            axes[row, col].axis('off')
        continue
    sample_idx = idx_candidates[0]
    img_batch = X_test_n[sample_idx:sample_idx + 1]
    cam = grad_cam(img_batch, label2idx[name])[0]

    axes[row, 0].imshow(X_test[sample_idx][:, :, 0], aspect='auto', origin='lower', cmap='viridis')
    axes[row, 0].set_title(f'{name} — Z channel (raw dB)')
    axes[row, 1].imshow(X_test[sample_idx][:, :, 0], aspect='auto', origin='lower', cmap='gray')
    axes[row, 1].imshow(cam, aspect='auto', origin='lower', cmap='jet', alpha=0.45,
                        extent=axes[row, 1].get_xlim() + axes[row, 1].get_ylim())
    axes[row, 1].set_title(f'{name} — Grad-CAM overlay')

plt.tight_layout()
gradcam_path = os.path.join(LOG_DIR, 'grad_cam_examples.png')
plt.savefig(gradcam_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'[SAVED] {gradcam_path}')

## Cell 15 — Save final model

In [ ]:
final_model_path = os.path.join(LOG_DIR, 'spectrogram_cnn_final.keras')
model.save(final_model_path)
print(f'[SAVED] {final_model_path}')
print(f'\nReload with:')
print(f"  model = tf.keras.models.load_model('{final_model_path}')")
print(f'\nDon\'t forget: apply the SAME normalization at inference time —')
print(f"  stats = np.load('{os.path.join(LOG_DIR, 'normalization_stats.npz')}')")
print(f"  X_new_norm = (X_new - stats['mean']) / stats['std']")